# Derived Features
This notebook engineers rolling 3-game average features from LeBron James' raw per-game stats. Raw source columns are removed after derivation, and the final dataset is saved as `LJ_Dataset_DERIV.csv`.

### Step 1: Load the Dataset
Load the cleaned LeBron James game log dataset (`LJ_Dataset_NODERIV.csv`). This dataset contains raw per-game stats such as `PTS`, `FGA`, `FG%`, `3PA`, `3P%`, `FTA`, `FT%`, and `MP` — which will be used to engineer rolling average features.

In [1]:
import pandas as pd

df = pd.read_csv('data/processed/LJ_Dataset_NODERIV.csv')
print(f'Shape before: {df.shape}')
df.head()

Shape before: (1421, 47)


,Date,Age,Home,MP,FGA,FG%,3PA,3P%,FTA,FT%,...,Opp_PHI,Opp_PHO,Opp_POR,Opp_SAC,Opp_SAS,Opp_SEA,Opp_TOR,Opp_UTA,Opp_WAS,HSG
0,2003-10-29,18.830137,0,42.833333,20,0.600,2,0.0,3,0.333,...,0,0,0,1,0,0,0,0,0,1
1,2003-10-30,18.832877,0,40.350000,17,0.471,5,0.2,7,0.571,...,0,1,0,0,0,0,0,0,0,0
2,2003-11-01,18.838356,0,39.166667,12,0.250,1,0.0,2,1.000,...,0,0,1,0,0,0,0,0,0,0
3,2003-11-05,18.849315,1,41.100000,11,0.273,2,0.0,1,1.000,...,0,0,0,0,0,0,0,0,0,0
4,2003-11-07,18.854795,0,43.733333,18,0.444,2,0.5,7,0.857,...,0,0,0,0,0,0,0,0,0,0


### Step 2: Engineer Rolling 3-Game Average Features
Instead of using raw per-game stats directly as model inputs, we derive rolling 3-game averages to capture recent form and trend leading into each game.

**Why rolling averages?**
A single game stat can be noisy (e.g., an outlier blowout or foul trouble). The last 3 games give a stable, recent picture of the player's current form.

**Why `shift(1)`?**
We shift by 1 before computing the rolling mean so that the average for game N is based on games N-3, N-2, N-1 — not game N itself. This prevents data leakage since the model should not know the current game's stats when predicting that game's outcome.

**Why `min_periods=3`?**
We require a full 3-game window. Rows with fewer than 3 prior games (i.e., the first 3 games of the dataset) will produce NaN and are dropped.

| Feature | Source | Description |
|---------|--------|-------------|
| `PPG3`  | `PTS`  | Avg points scored over the last 3 games |
| `3PA3`  | `3PA`  | Avg 3-point attempts over the last 3 games |
| `3P%3`  | `3P%`  | Avg 3-point shooting % over the last 3 games |
| `FGA3`  | `FGA`  | Avg field goal attempts over the last 3 games |
| `FG%3`  | `FG%`  | Avg field goal % over the last 3 games |
| `FTA3`  | `FTA`  | Avg free throw attempts over the last 3 games |
| `FT%3`  | `FT%`  | Avg free throw % over the last 3 games |
| `MP3`   | `MP`   | Avg minutes played over the last 3 games |

All values are rounded to 3 decimal places. The 8 raw source columns are removed after deriving the features.

In [2]:
raw_cols = ['PTS', '3PA', '3P%', 'FGA', 'FG%', 'FTA', 'FT%', 'MP']

df['PPG3']  = df['PTS'].shift(1).rolling(window=3, min_periods=3).mean().round(3)
df['3PA3']  = df['3PA'].shift(1).rolling(window=3, min_periods=3).mean().round(3)
df['3P%3']  = df['3P%'].shift(1).rolling(window=3, min_periods=3).mean().round(3)
df['FGA3']  = df['FGA'].shift(1).rolling(window=3, min_periods=3).mean().round(3)
df['FG%3']  = df['FG%'].shift(1).rolling(window=3, min_periods=3).mean().round(3)
df['FTA3']  = df['FTA'].shift(1).rolling(window=3, min_periods=3).mean().round(3)
df['FT%3']  = df['FT%'].shift(1).rolling(window=3, min_periods=3).mean().round(3)
df['MP3']   = df['MP'].shift(1).rolling(window=3, min_periods=3).mean().round(3)

df = df.dropna(subset=['PPG3']).reset_index(drop=True)
df = df.drop(columns=raw_cols)

print(f'Shape after: {df.shape}')
df.head()

Shape after: (1418, 47)


,Date,Age,Home,Opp_ATL,Opp_BOS,Opp_BRK,Opp_CHA,Opp_CHI,Opp_CHO,Opp_CLE,...,Opp_WAS,HSG,PPG3,3PA3,3P%3,FGA3,FG%3,FTA3,FT%3,MP3
0,2003-11-05,18.849315,1,0,0,0,0,0,0,0,...,0,0,18.000,2.667,0.067,16.333,0.440,4.000,0.635,40.783
1,2003-11-07,18.854795,0,0,0,0,0,0,0,0,...,0,0,12.000,2.667,0.067,13.333,0.331,3.333,0.857,40.206
2,2003-11-08,18.857534,1,0,0,0,0,0,0,0,...,1,0,12.667,1.667,0.167,13.667,0.322,3.333,0.952,41.333
3,2003-11-10,18.863014,1,0,0,0,0,0,0,0,...,0,0,15.667,1.333,0.167,16.000,0.379,4.000,0.702,43.111
4,2003-11-12,18.868493,0,0,0,0,0,0,0,0,...,0,0,19.000,1.667,0.500,16.333,0.483,3.667,0.369,40.628


### Step 3: Save the Derived Features Dataset
Export the final dataset to `LJ_Dataset_DERIV.csv`. This file contains all original columns except the 8 raw stat columns that were replaced by their rolling 3-game average counterparts. This dataset is ready to be used as input for model training.

In [3]:
df.to_csv('data/processed/LJ_Dataset_DERIV.csv', index=False)
print('Saved LJ_Dataset_DERIV.csv')
print(df.columns.tolist())

Saved LJ_Dataset_DERIV.csv
['Date', 'Age', 'Home', 'Opp_ATL', 'Opp_BOS', 'Opp_BRK', 'Opp_CHA', 'Opp_CHI', 'Opp_CHO', 'Opp_CLE', 'Opp_DAL', 'Opp_DEN', 'Opp_DET', 'Opp_GSW', 'Opp_HOU', 'Opp_IND', 'Opp_LAC', 'Opp_LAL', 'Opp_MEM', 'Opp_MIA', 'Opp_MIL', 'Opp_MIN', 'Opp_NJN', 'Opp_NOH', 'Opp_NOK', 'Opp_NOP', 'Opp_NYK', 'Opp_OKC', 'Opp_ORL', 'Opp_PHI', 'Opp_PHO', 'Opp_POR', 'Opp_SAC', 'Opp_SAS', 'Opp_SEA', 'Opp_TOR', 'Opp_UTA', 'Opp_WAS', 'HSG', 'PPG3', '3PA3', '3P%3', 'FGA3', 'FG%3', 'FTA3', 'FT%3', 'MP3']
